# Experimento 1 - Comparacao entre politicas

Objetivo: comparar `threshold`, `on_demand` e `hybrid`, implementadas no Controller como politicas distintas de atendimento de requisicao.


In [1]:
import sys
sys.path.insert(0, '/home/esdras/Wqnets/QuantumNet')

import pandas as pd
from quantumnet.topology import Network

print('Imports carregados com sucesso')

Imports carregados com sucesso


## Setup fixo do experimento

Para garantir comparacao justa entre politicas:
- Mesma topologia: `Linha` com 4 nos
- Mesmos pares Alice-Bob: `(0,1)`, `(1,2)`, `(2,3)`
- Mesma carga de requisicoes por par: `[32, 48, 64, 16, 40]`
- Mesmo threshold minimo por enlace: `96` bits

A unica variavel que muda e:
- `controller.set_policy("threshold")`
- `controller.set_policy("on_demand")`
- `controller.set_policy("hybrid")`


In [2]:
policies = ['threshold', 'on_demand', 'hybrid']
link_pairs = [(0, 1), (1, 2), (2, 3)]
request_load = [32, 48, 64, 16, 40]
minimum_stock_bits = 96

metrics_columns = [
    'total_requested_bits',
    'total_consumed_bits',
    'served_requests',
    'failed_requests',
    'denied_requests',
    'replenishment_events',
    'bits_available',
]

policy_link_rows = []

for policy in policies:
    exp_net = Network()
    exp_net.set_ready_topology('Linha', 4)
    exp_net.controller.set_policy(policy)

    for alice_id, bob_id in link_pairs:
        exp_net.controller.set_minimum_stock(alice_id, bob_id, minimum_stock_bits)

    for alice_id, bob_id in link_pairs:
        for requested_bits in request_load:
            exp_net.controller.handle_key_request(alice_id, bob_id, requested_bits)

    for alice_id, bob_id in link_pairs:
        state = exp_net.get_qkd_link_state(alice_id, bob_id)
        row = {
            'policy': policy,
            'link': f'({alice_id},{bob_id})',
        }
        for metric in metrics_columns:
            row[metric] = state[metric]
        policy_link_rows.append(row)

results_df = pd.DataFrame(policy_link_rows)
policy_summary = (
    results_df
    .groupby('policy', as_index=False)[metrics_columns]
    .sum()
    .sort_values('policy')
)

print('Resultado por enlace')
display(results_df.sort_values(['policy', 'link']).reset_index(drop=True))

print('\nResumo agregado por politica')
display(policy_summary)

print('\nLeitura rapida:')
for _, row in policy_summary.iterrows():
    print(
        f"- {row['policy']}: "
        f"served={row['served_requests']}, "
        f"failed={row['failed_requests']}, "
        f"denied={row['denied_requests']}, "
        f"consumed={row['total_consumed_bits']}, "
        f"replenishments={row['replenishment_events']}, "
        f"leftover={row['bits_available']}"
    )

Resultado por enlace


,policy,link,total_requested_bits,total_consumed_bits,served_requests,failed_requests,denied_requests,replenishment_events,bits_available
0,hybrid,"(0,1)",200,200,5,0,0,4,0
1,hybrid,"(1,2)",160,160,4,0,1,3,0
2,hybrid,"(2,3)",136,136,4,0,1,4,0
3,on_demand,"(0,1)",88,88,3,0,2,3,0
4,on_demand,"(1,2)",88,88,3,0,2,3,0
5,on_demand,"(2,3)",200,200,5,0,0,5,0
6,threshold,"(0,1)",96,96,3,0,2,1,0
7,threshold,"(1,2)",0,0,0,0,5,0,0
8,threshold,"(2,3)",0,0,0,0,5,0,0



Resumo agregado por politica


,policy,total_requested_bits,total_consumed_bits,served_requests,failed_requests,denied_requests,replenishment_events,bits_available
0,hybrid,496,496,13,0,2,11,0
1,on_demand,376,376,11,0,4,11,0
2,threshold,96,96,3,0,12,1,0



Leitura rapida:
- hybrid: served=13, failed=0, denied=2, consumed=496, replenishments=11, leftover=0
- on_demand: served=11, failed=0, denied=4, consumed=376, replenishments=11, leftover=0
- threshold: served=3, failed=0, denied=12, consumed=96, replenishments=1, leftover=0


## O que este experimento responde

A comparacao permite ver qual politica:
- atende mais requisicoes (`served_requests`);
- consome melhor o recurso-chave (`total_consumed_bits` em relacao a `total_requested_bits`);
- nega menos pedidos (`denied_requests`);
- depende mais de reposicao BB84 (`replenishment_events`).
